# Sommelier Kaggle Full Trace Run - HF API Overlap Standalone

Notebook này là bản single-file để chạy full pipeline trên Kaggle:
- Tự clone repo.
- Tự cài dependencies từ internet.
- Stage 3 dùng Hugging Face `audio_to_audio` API, không clone/tải SepReformer local.
- Không phụ thuộc notebook `00_build_sommelier_wheels_dataset.ipynb` hay private wheels dataset.

Điều kiện trước khi chạy:
- Kaggle Accelerator: GPU bật cho các stage nặng.
- Kaggle Internet: bật.
- Kaggle Secret có `HF_TOKEN`.
- Dataset audio đã Add Input vào notebook.

Lưu ý: vì đây là bản standalone, mỗi lần restart/đổi GPU có thể phải cài lại dependencies từ internet.


## 0. Cấu hình run

Chỉnh các biến bên dưới nếu muốn đổi branch, giới hạn thời lượng test, hoặc tắt bước nặng.

In [ ]:
REPO_URL = "https://github.com/lamkdhe180931-arch/sommelier.git"
BRANCH = "test-divide-stage"

RUN_DIR = "/kaggle/working/run_full"
INPUT_DIR = f"{RUN_DIR}/00_input"
DIAR_DIR = f"{RUN_DIR}/01_diarization"
MUSIC_DIR = f"{RUN_DIR}/02_music_clean"
OVERLAP_DIR = f"{RUN_DIR}/03_overlap"
ASR_DIR = f"{RUN_DIR}/04_asr"
EXPORT_DIR = f"{RUN_DIR}/05_export"
FINAL_DIR = f"{EXPORT_DIR}/final"
EVAL_DIR = f"{RUN_DIR}/06_eval"
PREVIEW_DIR = f"{RUN_DIR}/preview"
LOG_DIR_PATH = f"{RUN_DIR}/logs"
AUDIO_WAV = f"{INPUT_DIR}/full.wav"

# Để None nếu muốn chạy full audio. Để 300 nếu muốn test nhanh 5 phút.
AUDIO_LIMIT_SECONDS = 300

# Bước nặng. Có thể tắt để debug nhanh.
RUN_DEMUCS = True

# Stage 03 dùng Hugging Face audio-to-audio API thay SepReformer local.
# Model này tách speech source, không phải model tối ưu riêng tiếng Việt.
RUN_OVERLAP_API = True
OVERLAP_API_MODEL = "speechbrain/sepformer-whamr16k"
OVERLAP_API_PROVIDER = "hf-inference"
OVERLAP_API_TIMEOUT_SECONDS = 180
OVERLAP_API_FAIL_OPEN = True

# Guard cho overlap separation: tránh tách overlap quá ngắn gây méo tiếng/hallucination.
MIN_SEPREFORMER_OVERLAP_SECONDS = 1.0
MIN_SEPREFORMER_SEGMENT_SECONDS = 1.0

# Guard cho ASR ensemble: sửa các đoạn quá ngắn bị Whisper hallucinate.
ASR_QUALITY_GUARD = True
ASR_MICRO_SEGMENT_SECONDS = 0.5
ASR_SHORT_SEGMENT_SECONDS = 1.0
ASR_VI_AGREEMENT_THRESHOLD = 0.75

# ASRMoE chạy cả 3 model tiếng Việt: Whisper + PhoWhisper + ChunkFormer.
# Trên Kaggle 2xT4: Whisper đặt GPU0, PhoWhisper/ChunkFormer đặt GPU1.
ASR_MOE = True
WHISPER_DEVICE_INDEX = 0
VI_ASR_DEVICE_INDEX = 1
WHISPER_ARCH = "large-v3"
COMPUTE_TYPE = "float16"
ASR_THREADS = 4

# Thử chuyển PhoWhisper sang Hugging Face ASR API để giảm tải GPU/RAM local.
PHOWHISPER_USE_HF_API = True
PHOWHISPER_API_MODEL = "vinai/PhoWhisper-large"
PHOWHISPER_API_PROVIDER = "hf-inference"
PHOWHISPER_API_TIMEOUT_SECONDS = 180

HF_SECRET_NAME = "HF_TOKEN"


In [ ]:

from pathlib import Path
import os
import shlex
import subprocess

LOG_DIR = Path(LOG_DIR_PATH)
for _dir in [INPUT_DIR, DIAR_DIR, MUSIC_DIR, OVERLAP_DIR, ASR_DIR, EXPORT_DIR, FINAL_DIR, EVAL_DIR, PREVIEW_DIR, LOG_DIR_PATH]:
    Path(_dir).mkdir(parents=True, exist_ok=True)


def _format_cmd(cmd):
    if isinstance(cmd, (list, tuple)):
        return " ".join(shlex.quote(str(part)) for part in cmd)
    return str(cmd)


def tail_file(path, n=30):
    path = Path(path)
    if not path.exists():
        return ""
    lines = path.read_text(encoding="utf-8", errors="replace").splitlines()
    return "\n".join(lines[-n:])


def run_logged(cmd, log_name, cwd=None, env=None, shell=False, tail=20):
    log_path = LOG_DIR / log_name
    cwd = cwd or os.getcwd()
    print("Running:", _format_cmd(cmd))
    print("Log:", log_path)
    with open(log_path, "w", encoding="utf-8", errors="replace") as log:
        proc = subprocess.run(
            cmd,
            cwd=cwd,
            env=env,
            shell=shell,
            stdout=log,
            stderr=subprocess.STDOUT,
            text=True,
        )
    print("Exit code:", proc.returncode)
    if tail:
        log_tail = tail_file(log_path, n=tail)
        if log_tail:
            print(f"--- last {tail} log lines ---")
            print(log_tail)
    if proc.returncode != 0:
        raise subprocess.CalledProcessError(proc.returncode, cmd)
    return log_path


def export_audio_preview(audio_segment, out_path, seconds=30):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    audio_segment[: int(seconds * 1000)].export(out_path, format="wav")
    return out_path


## 1. Clone repo

In [ ]:

import os
import shutil
import subprocess
from pathlib import Path

os.chdir("/kaggle/working")
repo_dir = Path("/kaggle/working/sommelier")
if repo_dir.exists():
    shutil.rmtree(repo_dir)

run_logged(["git", "clone", "-b", BRANCH, REPO_URL, str(repo_dir)], "01_clone_repo.log", cwd="/kaggle/working", tail=20)
os.chdir(repo_dir / "podcast-pipeline")
print("cwd:", os.getcwd())
print("branch:", subprocess.check_output(["git", "rev-parse", "--abbrev-ref", "HEAD"], text=True).strip())
print("commit:", subprocess.check_output(["git", "log", "-1", "--oneline"], text=True).strip())


## 2. Cài dependencies từ internet

Bản standalone cài trực tiếp trong notebook này. Không cần private wheels dataset.


In [ ]:
import os
from pathlib import Path

os.chdir("/kaggle/working/sommelier/podcast-pipeline")

run_logged(["apt-get", "update", "-y"], "02_apt_update.log", tail=10)
run_logged(["apt-get", "install", "-y", "ffmpeg", "git", "git-lfs"], "03_apt_install.log", tail=10)
run_logged(["python", "-m", "pip", "install", "-U", "pip", "setuptools", "wheel", "packaging", "ninja"], "04_pip_base.log", tail=12)

req = Path("requirements.txt").read_text(encoding="utf-8")
filtered = [line for line in req.splitlines() if "nemo-toolkit[all]" not in line]
Path("requirements-kaggle.txt").write_text("\n".join(filtered) + "\n", encoding="utf-8")
run_logged(["python", "-m", "pip", "install", "-r", "requirements-kaggle.txt"], "05_pip_requirements.log", tail=20)

run_logged(["python", "-m", "pip", "uninstall", "-y", "nemo-toolkit", "lightning", "pytorch-lightning"], "06_pip_uninstall_nemo.log", tail=8)
run_logged(["python", "-m", "pip", "install", "lightning==2.4.0", "pytorch-lightning==2.5.2"], "07_pip_lightning.log", tail=12)
run_logged(["python", "-m", "pip", "install", "nemo-toolkit[asr]==2.4.0"], "08_pip_nemo_asr.log", tail=20)

run_logged(["python", "-m", "pip", "uninstall", "-y", "torch", "torchvision", "torchaudio"], "09_pip_uninstall_torch.log", tail=8)
run_logged([
    "python", "-m", "pip", "install", "--no-cache-dir", "--force-reinstall",
    "torch==2.7.1", "torchaudio==2.7.1", "torchvision==0.22.1",
    "--index-url", "https://download.pytorch.org/whl/cu126",
], "10_pip_torch_stack.log", tail=20)

run_logged(["python", "-m", "pip", "install", "pillow<12.0"], "11_pip_pillow.log", tail=8)
run_logged(["python", "-m", "pip", "install", "--no-cache-dir", "--force-reinstall", "--no-deps", "torchmetrics==1.7.4"], "12_pip_torchmetrics.log", tail=8)
run_logged([
    "python", "-m", "pip", "install", "--no-cache-dir", "--force-reinstall",
    "numpy==2.2.6", "numba==0.61.2", "llvmlite==0.44.0",
], "13_pip_numpy_numba.log", tail=12)

print("Dependency install logs saved in:", LOG_DIR)


## 3. Kiểm tra môi trường

In [ ]:
import importlib.metadata as importlib_metadata
import shutil
import subprocess
import numpy, numba, torch

if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=False)
else:
    print("nvidia-smi not found; kiểm tra torch.cuda bên dưới.")

print("nemo-toolkit:", importlib_metadata.version("nemo-toolkit"))
print("chunkformer:", importlib_metadata.version("chunkformer"))
print("numpy:", numpy.__version__)
print("numba:", numba.__version__)
print("torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print("GPU", i, torch.cuda.get_device_name(i))

import whisperx
print("whisperx ok")

import nemo.collections.asr as nemo_asr
print("nemo asr ok")

from nemo.collections.asr.models import SortformerEncLabelModel
print("sortformer import ok")


## 4. Gắn Hugging Face token vào config

In [ ]:
import json
from kaggle_secrets import UserSecretsClient
from huggingface_hub import whoami

token = UserSecretsClient().get_secret(HF_SECRET_NAME)
print("HF token:", token[:8] + "..." if token else "missing")
print(whoami(token=token))

with open("config.json", "r", encoding="utf-8") as f:
    cfg = json.load(f)

cfg["huggingface_token"] = token

with open("config.json", "w", encoding="utf-8") as f:
    json.dump(cfg, f, indent=2, ensure_ascii=False)

print("config.json updated")

## 5. Tìm audio input và chuẩn hóa audio

Output: `AUDIO_WAV = /kaggle/working/audio/full.wav`.

Chuẩn hóa về mono 16 kHz để các model dùng cùng format.

In [ ]:

from pathlib import Path

audio_exts = {".mp3", ".wav", ".m4a", ".flac", ".aac", ".ogg"}
audio_candidates = sorted(
    p for p in Path("/kaggle/input").rglob("*")
    if p.is_file() and p.suffix.lower() in audio_exts
)

if not audio_candidates:
    raise FileNotFoundError("Không tìm thấy audio trong /kaggle/input. Hãy Add Input hoặc Upload audio trước.")

AUDIO_IN = str(audio_candidates[0])
print("AUDIO_IN:", AUDIO_IN)
print("AUDIO_WAV:", AUDIO_WAV)
print("RUN_DIR:", RUN_DIR)

Path(INPUT_DIR).mkdir(parents=True, exist_ok=True)
Path(RUN_DIR).mkdir(parents=True, exist_ok=True)

cmd = ["ffmpeg", "-hide_banner", "-y", "-i", AUDIO_IN]
if AUDIO_LIMIT_SECONDS:
    cmd += ["-t", str(AUDIO_LIMIT_SECONDS)]
cmd += ["-ac", "1", "-ar", "16000", AUDIO_WAV]
run_logged(cmd, "00_prepare_audio_ffmpeg.log", cwd="/kaggle/working", tail=15)


In [ ]:

from pathlib import Path
from pydub import AudioSegment
from IPython.display import Audio, display

audio = AudioSegment.from_file(AUDIO_WAV)
print("Audio:", AUDIO_WAV)
print("Duration seconds:", len(audio) / 1000)
print("Frame rate:", audio.frame_rate)
print("Channels:", audio.channels)

preview_path = export_audio_preview(audio, Path(INPUT_DIR) / "preview_input_30s.wav", seconds=30)
print("Preview first 30s:", preview_path)
display(Audio(str(preview_path)))


## 6. Tải model phụ cho music clean và chuẩn bị overlap API

Bản này không clone SepReformer nữa, nên tránh lỗi Git LFS quota ở repo SepReformer.


In [ ]:
from huggingface_hub import hf_hub_download

if RUN_DEMUCS:
    panns_path = hf_hub_download(
        repo_id="thelou1s/panns-inference",
        filename="Cnn14_mAP=0.431.pth",
        local_dir="/kaggle/working/sommelier/panns_data",
    )
    print("PANNs checkpoint:", panns_path)
else:
    print("RUN_DEMUCS=False, bỏ qua tải PANNs")

In [ ]:
import os
from pathlib import Path

if RUN_OVERLAP_API:
    print("Stage 03 overlap separation dùng Hugging Face API")
    print("Model:", OVERLAP_API_MODEL)
    print("Provider:", OVERLAP_API_PROVIDER)
    print("Không clone/tải SepReformer local.")
else:
    print("RUN_OVERLAP_API=False, stage 3 chỉ ghi timeline segments, không tách overlap.")

Path(OVERLAP_DIR).mkdir(parents=True, exist_ok=True)
Path(f"{OVERLAP_DIR}/separated_segments").mkdir(parents=True, exist_ok=True)
os.chdir("/kaggle/working/sommelier/podcast-pipeline")


## 7. Trace VAD chunking

Bước này chỉ để xem VAD chia audio thành các chunk dài thế nào trước diarization. Đây không phải output speaker segment cuối cùng.

Output:
- `/kaggle/working/run_full/01_diarization/trace_vad_chunks.json`
- `/kaggle/working/run_full/01_diarization/vad_chunks/*.wav`

In [ ]:

import os
import json
import shutil
from pathlib import Path
import pandas as pd
from pydub import AudioSegment
from IPython.display import display

os.chdir("/kaggle/working/sommelier/podcast-pipeline")

import stage_common
import main_original_ASR_MoE as pipeline

cfg = pipeline.load_cfg("config.json")
logger = pipeline.Logger.get_logger()
pipeline.cfg = cfg
pipeline.logger = logger

device_name = "cuda" if pipeline.torch.cuda.is_available() else "cpu"
device = pipeline.torch.device(device_name)
pipeline.device_name = device_name
pipeline.device = device
pipeline.vad = pipeline.silero_vad.SileroVAD(device=device)

sample_rate = int(cfg["entrypoint"]["SAMPLE_RATE"])
audio_info = stage_common.load_audio_info(AUDIO_WAV, sample_rate)
diar_chunks, temp_chunk_dir = pipeline.prepare_diarization_chunks(AUDIO_WAV, audio_info)

chunk_dir = Path(DIAR_DIR) / "vad_chunks"
chunk_dir.mkdir(parents=True, exist_ok=True)

trace_chunks = []
for idx, chunk in enumerate(diar_chunks):
    src = Path(chunk["path"])
    dst = chunk_dir / f"chunk_{idx:03d}.wav"
    shutil.copy2(src, dst)
    duration = AudioSegment.from_file(dst).duration_seconds
    trace_chunks.append({
        "index": f"{idx:03d}",
        "path": str(dst),
        "offset": float(chunk["offset"]),
        "duration": float(duration),
        "start": float(chunk["offset"]),
        "end": float(chunk["offset"] + duration),
    })

if temp_chunk_dir:
    shutil.rmtree(temp_chunk_dir, ignore_errors=True)

stage_common.dump_json({
    "audio_path": AUDIO_WAV,
    "sample_rate": sample_rate,
    "chunks": trace_chunks,
    "metadata": {"stage": "vad_chunk_trace"},
}, Path(DIAR_DIR) / "trace_vad_chunks.json")

df_chunks = pd.DataFrame(trace_chunks)
print("VAD chunks:", len(df_chunks))
display(df_chunks.head(20))


In [ ]:

from pathlib import Path
from pydub import AudioSegment
from IPython.display import Audio, display

if trace_chunks:
    first = trace_chunks[0]
    print(first)
    chunk_audio = AudioSegment.from_file(first["path"])
    preview_path = export_audio_preview(chunk_audio, Path(DIAR_DIR) / "preview_vad_chunk_0_30s.wav", seconds=30)
    print("Preview first 30s of chunk 0:", preview_path)
    display(Audio(str(preview_path)))


## 8. Stage 01 - Speaker diarization

Output: `/kaggle/working/run_full/01_diarization/diarization.json`.

Đây là bước Sortformer + speaker linking, tạo segment có `start`, `end`, `speaker`.

In [ ]:

import os
os.chdir("/kaggle/working/sommelier/podcast-pipeline")

run_logged([
    "python", "stage_01_diarize.py",
    "--input_audio", AUDIO_WAV,
    "--out", f"{DIAR_DIR}/diarization.json",
    "--merge_gap", "2.0",
    "--max_segment_duration", "30.0",
    "--sortformer-pad-onset", "0.05",
    "--sortformer-pad-offset", "0.05",
], "18_stage_01_diarize.log", tail=25)


In [ ]:

import json
import pandas as pd
from IPython.display import display

with open(f"{DIAR_DIR}/diarization.json", "r", encoding="utf-8") as f:
    diar = json.load(f)

diar_segments = diar["segments"]
df_diar = pd.DataFrame(diar_segments)
df_diar["dur"] = df_diar["end"].astype(float) - df_diar["start"].astype(float)

print("File:", f"{DIAR_DIR}/diarization.json")
print("Total segments:", len(df_diar))
print("Speakers:", sorted(df_diar["speaker"].unique()) if len(df_diar) else [])
print("Duration median:", df_diar["dur"].median() if len(df_diar) else 0)
print("Duration mean:", df_diar["dur"].mean() if len(df_diar) else 0)
print("< 1s:", int((df_diar["dur"] < 1).sum()) if len(df_diar) else 0)
print("< 2s:", int((df_diar["dur"] < 2).sum()) if len(df_diar) else 0)
print("< 3s:", int((df_diar["dur"] < 3).sum()) if len(df_diar) else 0)

display(df_diar[["index", "start", "end", "dur", "speaker"]].head(30))


## 9. Stage 02 - Music/background clean

Output:
- `/kaggle/working/run_full/cleaned_audio.wav`
- `/kaggle/working/run_full/segment_flags.json`

In [ ]:

DEMUCS_ARG = "--demucs" if RUN_DEMUCS else "--no-demucs"

run_logged([
    "python", "stage_02_music_clean.py",
    "--input_audio", AUDIO_WAV,
    "--diarization_json", f"{DIAR_DIR}/diarization.json",
    "--out_audio", f"{MUSIC_DIR}/cleaned_audio.wav",
    "--out_flags", f"{MUSIC_DIR}/segment_flags.json",
    DEMUCS_ARG,
], "19_stage_02_music_clean.log", tail=25)


In [ ]:

from pathlib import Path
from pydub import AudioSegment
from IPython.display import Audio, display

with open(f"{MUSIC_DIR}/segment_flags.json", "r", encoding="utf-8") as f:
    flags_data = json.load(f)

flags = flags_data.get("segment_demucs_flags", [])
print("Cleaned audio:", flags_data["audio_path"])
print("Segments:", len(flags_data["segments"]))
print("Demucs flagged segments:", sum(bool(x) for x in flags), "/", len(flags))

cleaned_preview = export_audio_preview(AudioSegment.from_file(f"{MUSIC_DIR}/cleaned_audio.wav"), Path(MUSIC_DIR) / "preview_cleaned_30s.wav", seconds=30)
print("Preview first 30s:", cleaned_preview)
display(Audio(str(cleaned_preview)))


## 10. Stage 03 - Overlap separation bằng Hugging Face API

Output:
- `/kaggle/working/run_full/03_overlap/segments.json`
- `/kaggle/working/run_full/03_overlap/separated_segments/*.wav` nếu API tách được đoạn overlap.

Bước này chỉ gọi API cho các vùng overlap đủ dài theo diarization JSON, không tải SepReformer local.


In [ ]:

import contextlib
import io
import os
import tempfile
import time
import traceback
from pathlib import Path

import librosa
import numpy as np
import soundfile as sf
from huggingface_hub import InferenceClient

import stage_common
import main_original_ASR_MoE as pipeline


class HFAudioToAudioSeparator:
    """Adapter để dùng HF audio_to_audio API theo interface separator.separate()."""

    def __init__(self, model, token=None, provider=None, timeout=180):
        self.model = model
        self.provider = provider
        self.timeout = timeout
        self.client = self._build_client(token=token, provider=provider, timeout=timeout)

    @staticmethod
    def _build_client(token=None, provider=None, timeout=180):
        kwargs = {"timeout": timeout}
        if provider:
            kwargs["provider"] = provider
        if token:
            kwargs["api_key"] = token
        try:
            return InferenceClient(**kwargs)
        except TypeError:
            kwargs.pop("api_key", None)
            if token:
                kwargs["token"] = token
            try:
                return InferenceClient(**kwargs)
            except TypeError:
                kwargs.pop("provider", None)
                return InferenceClient(**kwargs)

    @staticmethod
    def _blob_from_item(item):
        if isinstance(item, (bytes, bytearray)):
            return bytes(item)
        if hasattr(item, "blob"):
            return item.blob
        if isinstance(item, dict):
            return item.get("blob") or item.get("content") or item.get("audio")
        return None

    @staticmethod
    def _decode_audio_blob(blob, sample_rate, tmp_dir, idx):
        try:
            wav, sr = sf.read(io.BytesIO(blob), dtype="float32", always_2d=False)
            if getattr(wav, "ndim", 1) > 1:
                wav = wav.mean(axis=1)
        except Exception:
            tmp_out = Path(tmp_dir) / f"hf_source_{idx}.wav"
            tmp_out.write_bytes(blob)
            wav, sr = librosa.load(str(tmp_out), sr=None, mono=True)

        wav = np.asarray(wav, dtype=np.float32)
        if sr != sample_rate:
            wav = librosa.resample(wav, orig_sr=sr, target_sr=sample_rate).astype(np.float32)
        return wav

    @staticmethod
    def _fit_length(wav, target_len):
        wav = np.asarray(wav, dtype=np.float32)
        if len(wav) < target_len:
            return np.pad(wav, (0, target_len - len(wav))).astype(np.float32)
        return wav[:target_len].astype(np.float32)

    def separate(self, audio_segment, sample_rate):
        audio_segment = np.asarray(audio_segment, dtype=np.float32)
        target_len = len(audio_segment)
        duration = target_len / float(sample_rate)
        print(f"Calling HF audio_to_audio: model={self.model}, duration={duration:.2f}s")

        with tempfile.TemporaryDirectory() as tmp_dir:
            input_path = Path(tmp_dir) / "overlap.wav"
            stage_common.write_wav(input_path, audio_segment, sample_rate)

            try:
                outputs = self.client.audio_to_audio(str(input_path), model=self.model)
            except TypeError:
                outputs = self.client.audio_to_audio(audio=str(input_path), model=self.model)

            if not isinstance(outputs, (list, tuple)) or len(outputs) < 2:
                raise RuntimeError(f"HF audio_to_audio cần trả về ít nhất 2 sources, nhận được: {type(outputs)}")

            sources = []
            for idx, item in enumerate(outputs[:2]):
                blob = self._blob_from_item(item)
                if not blob:
                    raise RuntimeError(f"HF output source {idx} không có audio blob")
                wav = self._decode_audio_blob(blob, sample_rate, tmp_dir, idx)
                sources.append(self._fit_length(wav, target_len))

        return sources[0], sources[1]


def _write_stage3_segments(segment_data, segments, audio_info, sample_rate, processing_time, enabled, error=None):
    separated_dir = Path(OVERLAP_DIR) / "separated_segments"
    separated_dir.mkdir(parents=True, exist_ok=True)

    clean_segments = []
    for idx, segment in enumerate(segments):
        clean_segment = stage_common.clean_segment_for_json(segment)
        if segment.get("sepreformer", False) and "enhanced_audio" in segment:
            seg_index = clean_segment.get("index") or stage_common.normalized_index(idx)
            speaker = clean_segment.get("speaker", "Unknown")
            enhanced_path = separated_dir / f"{seg_index}_{speaker}.wav"
            stage_common.write_wav(enhanced_path, segment["enhanced_audio"], sample_rate)
            clean_segment["enhanced_audio_path"] = str(enhanced_path.resolve())
            clean_segment["is_separated"] = True
            clean_segment["separation_method"] = "hf_audio_to_audio_api"
        else:
            clean_segment["is_separated"] = False
            clean_segment["separation_method"] = "none"
        clean_segments.append(clean_segment)

    stage_common.dump_json(
        {
            "audio_path": str(Path(MUSIC_DIR) / "cleaned_audio.wav"),
            "source_audio_path": segment_data.get("source_audio_path") or str(Path(MUSIC_DIR) / "cleaned_audio.wav"),
            "audio_name": segment_data.get("audio_name") or stage_common.audio_name_from_path(Path(MUSIC_DIR) / "cleaned_audio.wav"),
            "sample_rate": sample_rate,
            "audio_duration_seconds": len(audio_info["waveform"]) / sample_rate,
            "segments": clean_segments,
            "segment_demucs_flags": segment_data.get("segment_demucs_flags", [False] * len(clean_segments)),
            "metadata": {
                "stage": "overlap_separate",
                "enabled": bool(enabled),
                "separator": "hf_audio_to_audio_api" if enabled else "none",
                "api_model": OVERLAP_API_MODEL if enabled else "",
                "api_provider": OVERLAP_API_PROVIDER if enabled else "",
                "processing_time_seconds": processing_time,
                "overlap_threshold_seconds": 0.2,
                "min_sepreformer_overlap_seconds": MIN_SEPREFORMER_OVERLAP_SECONDS,
                "min_sepreformer_segment_seconds": MIN_SEPREFORMER_SEGMENT_SECONDS,
                "error": str(error) if error else "",
            },
        },
        Path(OVERLAP_DIR) / "segments.json",
    )


def run_stage03_overlap_api():
    cfg = pipeline.load_cfg("config.json")
    logger = pipeline.Logger.get_logger()
    pipeline.cfg = cfg
    pipeline.logger = logger

    segment_data = stage_common.load_json(f"{MUSIC_DIR}/segment_flags.json")
    segments = segment_data["segments"]
    sample_rate = int(segment_data.get("sample_rate") or cfg["entrypoint"]["SAMPLE_RATE"])
    audio_info = stage_common.load_audio_info(f"{MUSIC_DIR}/cleaned_audio.wav", sample_rate)

    device_name = "cuda" if pipeline.torch.cuda.is_available() else "cpu"
    device = pipeline.torch.device(device_name)
    pipeline.device_name = device_name
    pipeline.device = device

    start_time = time.time()
    error = None

    if RUN_OVERLAP_API:
        try:
            hf_token = cfg.get("huggingface_token") or globals().get("token")
            if not hf_token:
                raise RuntimeError("Thiếu HF token. Kiểm tra Kaggle Secret HF_TOKEN và cell gắn token vào config.json.")

            from pyannote.audio import Model as PyannoteModel

            embedding_model = PyannoteModel.from_pretrained(
                "pyannote/embedding",
                use_auth_token=hf_token,
            ).to(device)
            separator = HFAudioToAudioSeparator(
                model=OVERLAP_API_MODEL,
                token=hf_token,
                provider=OVERLAP_API_PROVIDER,
                timeout=OVERLAP_API_TIMEOUT_SECONDS,
            )
            audio_info, segments = pipeline.process_overlapping_segments_with_separation(
                segments,
                audio_info,
                overlap_threshold=0.2,
                separator=separator,
                embedding_model=embedding_model,
                min_sepreformer_overlap=MIN_SEPREFORMER_OVERLAP_SECONDS,
                min_sepreformer_segment=MIN_SEPREFORMER_SEGMENT_SECONDS,
            )
        except Exception as exc:
            error = exc
            print("Stage 03 HF API failed:", repr(exc))
            if not OVERLAP_API_FAIL_OPEN:
                raise
            print("OVERLAP_API_FAIL_OPEN=True nên ghi segments.json không tách overlap để stage sau chạy tiếp.")
    else:
        print("RUN_OVERLAP_API=False, bỏ qua tách overlap và chỉ ghi segments.json.")

    processing_time = time.time() - start_time
    _write_stage3_segments(segment_data, segments, audio_info, sample_rate, processing_time, RUN_OVERLAP_API and error is None, error=error)
    print("Stage 03 complete:", Path(OVERLAP_DIR) / "segments.json")


log_path = Path(LOG_DIR_PATH) / "20_stage_03_overlap_separate.log"
log_path.parent.mkdir(parents=True, exist_ok=True)
with open(log_path, "w", encoding="utf-8") as log_file:
    with contextlib.redirect_stdout(log_file), contextlib.redirect_stderr(log_file):
        try:
            run_stage03_overlap_api()
        except Exception:
            traceback.print_exc()
            raise

print("Log:", log_path)
print(tail_file(log_path, 80))


In [ ]:
with open(f"{OVERLAP_DIR}/segments.json", "r", encoding="utf-8") as f:
    seg_data = json.load(f)

segments = seg_data["segments"]
df_seg = pd.DataFrame(segments)
df_seg["dur"] = df_seg["end"].astype(float) - df_seg["start"].astype(float)

print("Segments:", len(df_seg))
print("Separated by API:", int(df_seg.get("is_separated", pd.Series(dtype=bool)).fillna(False).sum()) if len(df_seg) else 0)
print("Duration median:", df_seg["dur"].median() if len(df_seg) else 0)
display(df_seg[[c for c in ["index", "start", "end", "dur", "speaker", "is_separated", "separation_method", "enhanced_audio_path"] if c in df_seg.columns]].head(30))


In [ ]:
from pathlib import Path
import json
import pandas as pd
from pydub import AudioSegment
from IPython.display import Audio, display, HTML

segments_path = Path(OVERLAP_DIR) / "segments.json"
cleaned_audio_path = Path(MUSIC_DIR) / "cleaned_audio.wav"

with open(segments_path, "r", encoding="utf-8") as f:
    stage3 = json.load(f)

segments = stage3["segments"]
cleaned_audio = AudioSegment.from_file(cleaned_audio_path)

df = pd.DataFrame([
    {
        "i": i,
        "start": s.get("start"),
        "end": s.get("end"),
        "speaker": s.get("speaker"),
        "is_separated": s.get("is_separated", False),
        "separation_method": s.get("separation_method", ""),
        "enhanced_audio_path": s.get("enhanced_audio_path", ""),
    }
    for i, s in enumerate(segments)
])

print("Stage 03 segments:", len(df))
print("Separated segments:", int(df["is_separated"].sum()) if len(df) else 0)
display(df.head(20))


def listen_stage3_segment(i):
    s = segments[i]
    start = float(s["start"])
    end = float(s["end"])
    speaker = s.get("speaker", "UNKNOWN")
    is_separated = bool(s.get("is_separated", False))
    enhanced_path = s.get("enhanced_audio_path")

    print(f"Segment {i}")
    print(f"Speaker: {speaker}")
    print(f"Time: {start:.2f}s - {end:.2f}s")
    print(f"API separated: {is_separated}")

    tmp_clip = Path(PREVIEW_DIR) / f"preview_stage3_cleaned_{i:04d}.wav"
    cleaned_audio[int(start * 1000):int(end * 1000)].export(tmp_clip, format="wav")

    display(HTML("<b>Cleaned audio slice sau Stage 02/03 timeline:</b>"))
    display(Audio(str(tmp_clip)))

    if enhanced_path and Path(enhanced_path).exists():
        display(HTML("<b>Enhanced overlap API segment:</b>"))
        display(Audio(enhanced_path))
    else:
        print("Không có enhanced_audio_path cho segment này.")

print("Call listen_stage3_segment(i) để nghe thủ công, ví dụ: listen_stage3_segment(0)")


## 11. Cài cuDNN 8 riêng cho faster-whisper/ctranslate2

Không cài đè vào global torch. Chỉ cài vào `/kaggle/working/cudnn8` rồi truyền `LD_LIBRARY_PATH` khi chạy ASR.


In [ ]:
import shutil
from pathlib import Path

cudnn_dir = Path("/kaggle/working/cudnn8")
if cudnn_dir.exists():
    shutil.rmtree(cudnn_dir)

run_logged([
    "python", "-m", "pip", "install", "--target", str(cudnn_dir),
    "nvidia-cudnn-cu12==8.9.7.29",
], "21_pip_cudnn8.log", cwd="/kaggle/working/sommelier/podcast-pipeline", tail=20)

matches = sorted(cudnn_dir.rglob("libcudnn_ops_infer.so.8"))
print("libcudnn matches:", len(matches))
for p in matches[:5]:
    print(p)


## 11.5 Patch PhoWhisper sang Hugging Face API

Cell này patch runtime module `models/vietnamese_asr.py` trong repo đã clone. Không sửa repo gốc; chỉ áp dụng cho session Kaggle hiện tại.


In [ ]:
from pathlib import Path

asr_module_path = Path("/kaggle/working/sommelier/podcast-pipeline/models/vietnamese_asr.py")
src = asr_module_path.read_text(encoding="utf-8")

api_class = '''

class PhoWhisperHFAPITranscriber:
    def __init__(self, model_name: str = PHOWHISPER_MODEL_NAME, provider=None, timeout: int = 180, token=None) -> None:
        self.model_name = model_name
        self.provider = provider or None
        self.timeout = timeout
        self.token = token
        self.client = self._build_client()

    def _build_client(self):
        from huggingface_hub import InferenceClient

        kwargs = {"timeout": self.timeout}
        if self.provider:
            kwargs["provider"] = self.provider
        if self.token:
            kwargs["api_key"] = self.token
        try:
            return InferenceClient(**kwargs)
        except TypeError:
            kwargs.pop("api_key", None)
            if self.token:
                kwargs["token"] = self.token
            try:
                return InferenceClient(**kwargs)
            except TypeError:
                kwargs.pop("provider", None)
                return InferenceClient(**kwargs)

    def transcribe(self, audio_16k) -> str:
        import tempfile
        import soundfile as sf

        audio = _as_float32_mono(audio_16k)
        with tempfile.NamedTemporaryFile(suffix=".wav", delete=True) as temp_wav:
            sf.write(temp_wav.name, audio, SAMPLE_RATE)
            temp_wav.flush()
            try:
                result = self.client.automatic_speech_recognition(temp_wav.name, model=self.model_name)
            except TypeError:
                result = self.client.automatic_speech_recognition(audio=temp_wav.name, model=self.model_name)
        return extract_transcription_text(result)
'''

if "class PhoWhisperHFAPITranscriber" not in src:
    src = src.replace("\ndef load_phowhisper_model", api_class + "\ndef load_phowhisper_model", 1)

old_loader = """def load_phowhisper_model(device=None) -> PhoWhisperTranscriber:
    return PhoWhisperTranscriber.from_pretrained(device=device)
"""
new_loader = """def load_phowhisper_model(device=None):
    import os

    use_api = os.environ.get("PHOWHISPER_USE_HF_API", "0").strip().lower() in {"1", "true", "yes", "on"}
    if use_api:
        timeout = int(os.environ.get("PHOWHISPER_API_TIMEOUT_SECONDS", "180"))
        token = os.environ.get("PHOWHISPER_HF_TOKEN") or os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACEHUB_API_TOKEN")
        return PhoWhisperHFAPITranscriber(
            model_name=os.environ.get("PHOWHISPER_API_MODEL", PHOWHISPER_MODEL_NAME),
            provider=os.environ.get("PHOWHISPER_API_PROVIDER", "hf-inference"),
            timeout=timeout,
            token=token,
        )
    return PhoWhisperTranscriber.from_pretrained(device=device)
"""

if old_loader in src:
    src = src.replace(old_loader, new_loader)
elif "PHOWHISPER_USE_HF_API" not in src:
    raise RuntimeError("Không tìm thấy load_phowhisper_model để patch.")

asr_module_path.write_text(src, encoding="utf-8")
print("Patched PhoWhisper loader:", asr_module_path)
print("PHOWHISPER_USE_HF_API:", PHOWHISPER_USE_HF_API)
print("PHOWHISPER_API_MODEL:", PHOWHISPER_API_MODEL)


## 12. Stage 04 - ASRMoE tiếng Việt

Output: `/kaggle/working/run_full/04_asr/transcript.json`.

Bản này chạy:
- Whisper large-v3 local trên GPU `WHISPER_DEVICE_INDEX`.
- PhoWhisper qua Hugging Face ASR API nếu `PHOWHISPER_USE_HF_API=True`.
- ChunkFormer local trên GPU `VI_ASR_DEVICE_INDEX`.

Nếu HF API PhoWhisper lỗi, segment đó trả text rỗng cho `text_phowhisper`, còn Whisper/ChunkFormer vẫn chạy tiếp.


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=False)
else:
    print("nvidia-smi not found; tiếp tục nếu torch CUDA vẫn khả dụng.")

env = os.environ.copy()
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
extra_ld_paths = [
    "/kaggle/working/cudnn8/nvidia/cudnn/lib",
    "/kaggle/working/cudnn8/nvidia/cublas/lib",
    "/kaggle/working/cudnn8/nvidia/cuda_nvrtc/lib",
]
extra_ld_paths = [p for p in extra_ld_paths if Path(p).exists()]
env["LD_LIBRARY_PATH"] = ":".join(extra_ld_paths + [env.get("LD_LIBRARY_PATH", "")]).rstrip(":")
print("extra LD paths:", extra_ld_paths)

env["PHOWHISPER_USE_HF_API"] = "1" if PHOWHISPER_USE_HF_API else "0"
env["PHOWHISPER_API_MODEL"] = PHOWHISPER_API_MODEL
env["PHOWHISPER_API_PROVIDER"] = PHOWHISPER_API_PROVIDER
env["PHOWHISPER_API_TIMEOUT_SECONDS"] = str(PHOWHISPER_API_TIMEOUT_SECONDS)
if "token" in globals() and token:
    env["HF_TOKEN"] = token
    env["PHOWHISPER_HF_TOKEN"] = token
print("PhoWhisper API:", env["PHOWHISPER_USE_HF_API"], env["PHOWHISPER_API_MODEL"])

cmd = [
    "python", "stage_04_asr.py",
    "--segments_json", f"{OVERLAP_DIR}/segments.json",
    "--audio", f"{MUSIC_DIR}/cleaned_audio.wav",
    "--out", f"{ASR_DIR}/transcript.json",
    "--ASRMoE" if ASR_MOE else "--no-ASRMoE",
    "--no-whisperx_word_timestamps",
    "--no-initprompt",
    "--whisper_arch", WHISPER_ARCH,
    "--compute_type", COMPUTE_TYPE,
    "--threads", str(ASR_THREADS),
    "--whisper_device_index", str(WHISPER_DEVICE_INDEX),
    "--vi_asr_device_index", str(VI_ASR_DEVICE_INDEX),
    "--asr_quality_guard" if ASR_QUALITY_GUARD else "--no-asr_quality_guard",
    "--asr_micro_segment_seconds", str(ASR_MICRO_SEGMENT_SECONDS),
    "--asr_short_segment_seconds", str(ASR_SHORT_SEGMENT_SECONDS),
    "--asr_vi_agreement_threshold", str(ASR_VI_AGREEMENT_THRESHOLD),
]
run_logged(cmd, "22_stage_04_asr.log", cwd="/kaggle/working/sommelier/podcast-pipeline", env=env, tail=30)


In [ ]:

with open(f"{ASR_DIR}/transcript.json", "r", encoding="utf-8") as f:
    transcript = json.load(f)

tr_segments = transcript["segments"]
print("Transcript segments:", len(tr_segments))
print("Metadata keys:", sorted(transcript.get("metadata", {}).keys()))

for s in tr_segments[:30]:
    start = float(s.get("start", 0))
    end = float(s.get("end", 0))
    speaker = s.get("speaker", "UNKNOWN")
    text = s.get("text", "").strip()
    print(f"[{start:07.2f} - {end:07.2f}] {speaker}: {text}")
 
if len(tr_segments) > 30:
    print(f"... hidden {len(tr_segments) - 30} more segments. Full JSON: {ASR_DIR}/transcript.json")


## 13. Stage 05 - Export final JSON và audio segment MP3

Output:
- `/kaggle/working/run_full/final/data_audio.json`
- `/kaggle/working/run_full/final/data_audio/*.mp3`

In [ ]:

run_logged([
    "python", "stage_05_export.py",
    "--transcript_json", f"{ASR_DIR}/transcript.json",
    "--audio", f"{MUSIC_DIR}/cleaned_audio.wav",
    "--out_dir", FINAL_DIR,
    "--audio_name", "data_audio",
], "23_stage_05_export.log", cwd="/kaggle/working/sommelier/podcast-pipeline", tail=25)

files = sorted(Path(RUN_DIR).glob("**/*"))
files = [p for p in files if p.is_file()]
print("Files:", len(files))
for p in files[:50]:
    print(p)
if len(files) > 50:
    print(f"... hidden {len(files) - 50} more files")


## 14. Review outputs by stage


In [ ]:

from pathlib import Path
import json
import pandas as pd
from pydub import AudioSegment
from IPython.display import Audio, display

STAGE_LOGS = {
    "0": ["00_prepare_audio_ffmpeg.log"],
    "audio": ["00_prepare_audio_ffmpeg.log"],
    "1": ["18_stage_01_diarize.log"],
    "01": ["18_stage_01_diarize.log"],
    "diarization": ["18_stage_01_diarize.log"],
    "2": ["19_stage_02_music_clean.log"],
    "02": ["19_stage_02_music_clean.log"],
    "music": ["19_stage_02_music_clean.log"],
    "3": ["20_stage_03_overlap_separate.log"],
    "03": ["20_stage_03_overlap_separate.log"],
    "overlap": ["20_stage_03_overlap_separate.log"],
    "4": ["22_stage_04_asr.log"],
    "04": ["22_stage_04_asr.log"],
    "asr": ["22_stage_04_asr.log"],
    "5": ["23_stage_05_export.log"],
    "05": ["23_stage_05_export.log"],
    "export": ["23_stage_05_export.log"],
    "6": ["24_stage_06_eval.log"],
    "06": ["24_stage_06_eval.log"],
    "eval": ["24_stage_06_eval.log"],
    "install": [
        "02_apt_update.log", "03_apt_install.log", "04_pip_base.log",
        "05_pip_requirements.log", "08_pip_nemo_asr.log", "10_pip_torch_stack.log",
        "12_pip_torchmetrics.log", "13_pip_numpy_numba.log", "21_pip_cudnn8.log",
    ],
}


def _stage_key(stage):
    return str(stage).lower().replace("stage", "").replace(" ", "").strip()


def show_stage_log(stage, lines=25):
    key = _stage_key(stage)
    logs = STAGE_LOGS.get(key, [])
    if not logs:
        print("Unknown stage. Try: 1, 2, 3, 4, 5, 6, install, audio")
        return

    for name in logs:
        path = Path(LOG_DIR_PATH) / name
        print("\n" + "=" * 90)
        print(path)
        print("=" * 90)
        if path.exists():
            print(tail_file(path, lines))
        else:
            print("Log chưa tồn tại. Hãy chạy stage tương ứng trước.")


def _load_json(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Chưa có file: {path}")
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def _table(records, columns=None, n=5):
    df = pd.DataFrame(records[:n])
    if columns:
        columns = [c for c in columns if c in df.columns]
        df = df[columns]
    display(df)
    return df


def _slice_audio(audio_path, start, end, out_name, pad=0.15, max_seconds=45):
    audio_path = Path(audio_path)
    if not audio_path.exists():
        print("Missing audio:", audio_path)
        return None

    start = max(0.0, float(start) - pad)
    end = max(start, float(end) + pad)
    if end - start > max_seconds:
        end = start + max_seconds

    audio = AudioSegment.from_file(audio_path)
    out = Path(PREVIEW_DIR) / out_name
    out.parent.mkdir(parents=True, exist_ok=True)
    audio[int(start * 1000):int(end * 1000)].export(out, format="wav")
    return out


def _show_segment_audio(segments, audio_path, n=5, pad=0.15, prefer_enhanced=True):
    for i, seg in enumerate(segments[:n]):
        print("\n--- audio", i, "---")
        print(f"[{float(seg.get('start', 0)):.2f} - {float(seg.get('end', 0)):.2f}]", seg.get("speaker", ""))

        enhanced = seg.get("enhanced_audio_path") if prefer_enhanced else None
        if enhanced and Path(enhanced).exists():
            print("enhanced:", enhanced)
            display(Audio(enhanced))
            continue

        out = _slice_audio(
            audio_path,
            seg.get("start", 0),
            seg.get("end", 0),
            f"_review_stage_audio_{i:03d}.wav",
            pad=pad,
        )
        if out:
            display(Audio(str(out)))


def review_stage(stage, n=5, log_lines=25, play_audio=True, pad=0.15):
    """Review compact một stage: log ngắn + 5 dòng đầu + tối đa 5 audio clip."""
    key = _stage_key(stage)
    print("Review stage:", stage)
    show_stage_log(key, lines=log_lines)

    if key in {"1", "01", "diarization"}:
        data = _load_json(Path(DIAR_DIR) / "diarization.json")
        segments = data.get("segments", [])
        print("\nDiarization segments:", len(segments))
        _table(segments, ["index", "start", "end", "speaker"], n=n)
        if play_audio:
            _show_segment_audio(segments, AUDIO_WAV, n=n, pad=pad, prefer_enhanced=False)
        return

    if key in {"2", "02", "music"}:
        data = _load_json(Path(MUSIC_DIR) / "segment_flags.json")
        segments = data.get("segments", [])
        flags = data.get("segment_demucs_flags", [])
        rows = []
        for i, seg in enumerate(segments):
            row = dict(seg)
            row["demucs_flag"] = bool(flags[i]) if i < len(flags) else False
            rows.append(row)
        print("\nMusic-clean segments:", len(rows))
        print("Demucs flagged:", sum(bool(x) for x in flags), "/", len(flags))
        _table(rows, ["index", "start", "end", "speaker", "demucs_flag"], n=n)
        if play_audio:
            _show_segment_audio(rows, Path(MUSIC_DIR) / "cleaned_audio.wav", n=n, pad=pad, prefer_enhanced=False)
        return

    if key in {"3", "03", "overlap"}:
        data = _load_json(Path(OVERLAP_DIR) / "segments.json")
        segments = data.get("segments", [])
        print("\nStage 03 segments:", len(segments))
        _table(segments, ["index", "start", "end", "speaker", "is_separated", "separation_method", "sepreformer_skip_reasons", "enhanced_audio_path"], n=n)
        if play_audio:
            _show_segment_audio(segments, Path(MUSIC_DIR) / "cleaned_audio.wav", n=n, pad=pad, prefer_enhanced=True)
        return

    if key in {"4", "04", "asr"}:
        data = _load_json(Path(ASR_DIR) / "transcript.json")
        segments = data.get("segments", [])
        print("\nTranscript segments:", len(segments))
        _table(
            segments,
            ["index", "start", "end", "speaker", "text", "asr_quality_source", "asr_quality_actions", "text_whisper", "text_phowhisper", "text_chunkformer"],
            n=n,
        )
        if play_audio:
            _show_segment_audio(segments, Path(MUSIC_DIR) / "cleaned_audio.wav", n=n, pad=pad, prefer_enhanced=True)
        return

    if key in {"5", "05", "export"}:
        final_dir = Path(FINAL_DIR) / "data_audio"
        files = sorted(final_dir.glob("*.mp3"))
        print("\nExported MP3:", len(files))
        display(pd.DataFrame({"i": range(min(n, len(files))), "path": [str(p) for p in files[:n]]}))
        if play_audio:
            for i, path in enumerate(files[:n]):
                print("\n--- exported", i, "---")
                print(path)
                display(Audio(str(path)))
        return

    print("Stage này chỉ có log. Dùng show_stage_log(stage) để xem log.")


print("Dùng: review_stage(1), review_stage(2), review_stage(3), review_stage(4), review_stage(5), review_stage(6)")
print("Ví dụ chỉ xem log stage 4: show_stage_log(4, lines=80)")
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 0)
# review_stage(1)
review_stage(4, n=10, play_audio=True)

## 15. Stage 06 - Eval metrics

Tinh metric sau khi chay xong cac stage: audio info, segment stats, ASR suspicious segments, export consistency va recommendation.

In [ ]:

from pathlib import Path
from IPython.display import Markdown, display

run_logged([
    "python", "stage_06_eval.py",
    "--run_dir", RUN_DIR,
    "--print_markdown",
], "24_stage_06_eval.log", cwd="/kaggle/working/sommelier/podcast-pipeline", tail=80)

report_md = Path(EVAL_DIR) / "eval_report.md"
report_json = Path(EVAL_DIR) / "eval_report.json"
print("Eval markdown:", report_md)
print("Eval json:", report_json)

if report_md.exists():
    display(Markdown(report_md.read_text(encoding="utf-8")))
else:
    print("Eval report chua ton tai. Kiem tra log stage 06.")


In [ ]:
import shutil
from pathlib import Path
from IPython.display import FileLink, display

run_dir = Path(RUN_DIR)
zip_base = Path("/kaggle/working/run_full_download")

zip_path = shutil.make_archive(
    base_name=str(zip_base),
    format="zip",
    root_dir=str(run_dir.parent),
    base_dir=run_dir.name,
)

print("Created:", zip_path)
display(FileLink(zip_path))